importing the transformers library

In [27]:

from transformers import pipeline
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
from datasets import Dataset
import os
from transformers import AutoTokenizer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch

torch.cuda.empty_cache()


importing the language model and the tokenizer
code just outputs the language codes in the model
(contains tagalog and english)

In [17]:
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-tl")
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-tl")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


In [18]:
src_texts = ["I am a small frog."]

inputs = tokenizer(src_texts, return_tensors="pt", padding=True).to(device)

outputs = model.generate(**inputs)
print(outputs)

translated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for src, tgt in zip(src_texts, translated_texts):
    print(f"EN: {src}")
    print(f"TL: {tgt}\n")

tensor([[57372,   785,    20,    34,    25,  1147,     6, 14708,     3,     0]],
       device='cuda:0')
EN: I am a small frog.
TL: Ako'y isang maliit na palaka.



testing translation from en -> tl

In [19]:
# Read all text files
def load_parallel_files(file_paths):
    data = {}
    
    for lang_code, filepath in file_paths.items():
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines()]
            data[lang_code] = lines
    
    return data

# Define the folder and files
data_folder = "parallel-corpora"
file_paths = {
    'en': os.path.join(data_folder, "eng.txt"),
    'tl': os.path.join(data_folder, "tgl.txt"), 
    'pam': os.path.join(data_folder, "pam.txt"),
    'tao': os.path.join(data_folder, "tao.txt"),
    'pag': os.path.join(data_folder, "pag.txt")
}

parallel_data = load_parallel_files(file_paths)

print(parallel_data['en'])
print(parallel_data['tl'])
print(parallel_data['pam'])
print(parallel_data['tao'])
print(parallel_data['pag'])

['An account of the genealogy of Jesus the Messiah, the son of David, the son of Abraham.', 'Abraham was the father of Isaac, and Isaac the father of Jacob, and Jacob the father of Judah and his brothers,', 'and Judah the father of Perez and Zerah by Tamar, and Perez the father of Hezron, and Hezron the father of Aram,', 'and Aram the father of Aminadab, and Aminadab the father of Nahshon, and Nahshon the father of Salmon,', 'and Salmon the father of Boaz by Rahab, and Boaz the father of Obed by Ruth, and Obed the father of Jesse,', 'and Jesse the father of King David.And David was the father of Solomon by the wife of Uriah,', 'and Solomon the father of Rehoboam, and Rehoboam the father of Abijah, and Abijah the father of Asaph,', 'and Asaph the father of Jehoshaphat, and Jehoshaphat the father of Joram, and Joram the father of Uzziah,', 'and Uzziah the father of Jotham, and Jotham the father of Ahaz, and Ahaz the father of Hezekiah,', 'and Hezekiah the father of Manasseh, and Manasseh

In [20]:
parallel_data_dict = {
    'en':parallel_data['en'],
    'tl':parallel_data['tl'],
    'pam':parallel_data['pam'],
    'tao':parallel_data['tao'],
    'pag':parallel_data['pag']
}

nSentences = len(parallel_data['en'])

print(nSentences)

lang_data = []

langauge_list = ['en', 'tl', 'pam', 'tao', 'pag']

for i in range (nSentences):
    for source in langauge_list:
        for dest in langauge_list:
            if source != dest:
                lang_data.append({
                    'source_language': source,
                    'dest_language': dest,
                    'source_text': parallel_data[source][i],
                    'dest_text': parallel_data[dest][i]
                })

print(lang_data[0])

lang_df = pd.DataFrame(lang_data, columns = ['source_language', 'dest_language', 'source_text', 'dest_text'])

3743
{'source_language': 'en', 'dest_language': 'tl', 'source_text': 'An account of the genealogy of Jesus the Messiah, the son of David, the son of Abraham.', 'dest_text': 'Ang aklat ng lahi ni Jesucristo, na anak ni David, na anak ni Abraham.'}


CONVERT PANDAS DATAFRAME TO HUGGINGFACE DATASET

In [21]:
hf_data = Dataset.from_pandas(lang_df)

training_data = hf_data.train_test_split(test_size=0.15)

training_data['test']

Dataset({
    features: ['source_language', 'dest_language', 'source_text', 'dest_text'],
    num_rows: 11229
})

In [22]:
new_langs = ["__pam__", "__tao__", "__pag__"]
tokenizer.add_special_tokens({'additional_special_tokens': new_langs})
model.resize_token_embeddings(len(tokenizer))

pam_id = tokenizer.convert_tokens_to_ids("__pam__")
tao_id = tokenizer.convert_tokens_to_ids("__tao__") 
pag_id = tokenizer.convert_tokens_to_ids("__pag__")


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
prefix = "translating"

def preprocessing(training_data):
    training_inputs = []
    training_labels = []

    for src, dst, src_text, dst_text in zip(training_data['source_language'], training_data['dest_language'], training_data['source_text'], training_data['dest_text']):
        src_with_lang = f">>{src}<< {src_text}"
        dst_with_lang = dst_text

        tokenized_inputs = tokenizer(src_with_lang, return_tensors="pt", padding=True,  truncation=True, max_length=128).to(device)
        target = tokenizer(dst_with_lang, return_tensors="pt", padding=True,  truncation=True, max_length=128).to(device) 
        
        # Extract the integer lists from tensors
        training_inputs.append(tokenized_inputs['input_ids'][0].tolist())
        training_labels.append(target['input_ids'][0].tolist())

    return {
        'input_ids': training_inputs,
        'labels': training_labels
    }

tokenized_data = training_data.map(preprocessing, batched=True)

Map:   0%|          | 0/63631 [00:00<?, ? examples/s]

Map:   0%|          | 0/11229 [00:00<?, ? examples/s]

In [24]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model)

model.gradient_checkpointing_enable()

model.gradient_checkpointing_enable()
training_args = Seq2SeqTrainingArguments(
    output_dir="./trained-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=False,  # This fixes the speed issue
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [25]:
print(f"Model device: {next(model.parameters()).device}")

# Monitor GPU usage
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f}GB")

model.cuda()
print(f"Model device: {next(model.parameters()).device}")

Model device: cuda:0
GPU memory: 0.28GB / 6.00GB
Model device: cuda:0


In [ ]:
'''trainer.train()

model.save_pretrained("./pretrained-model")
tokenizer.save_pretrained("./finetuned-tokenizer")'''

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,2.909100,2.642783
2,2.638100,2.465887


c:\Users\Coco\anaconda3\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[57372]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('./finetuned-tokenizer\\tokenizer_config.json',
 './finetuned-tokenizer\\special_tokens_map.json',
 './finetuned-tokenizer\\vocab.json',
 './finetuned-tokenizer\\source.spm',
 './finetuned-tokenizer\\target.spm',
 './finetuned-tokenizer\\added_tokens.json')

In [42]:
def translate_to_language(text, target_language):
    """
    Translate text to the specified target language
    
    target_language: one of ['pam', 'tao', 'pag', 'tl', 'en']
    """
    # Format the source text with language token (same as during training)
    source_text = f"__{target_language}__ {text}"
    
    # Tokenize
    inputs = tokenizer(source_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    
    # Generate translation
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            early_stopping=True,
            temperature=0.7
        )
    
    # Decode and remove special tokens
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

# Alternative function that matches your training format more closely
def translate_between_languages(text, source_lang, target_lang):
    """
    Translate from source language to target language
    """
    source_text = f"__{target_lang}__ {text}"  # Note: Based on your training, target lang goes in source!
    
    inputs = tokenizer(source_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            early_stopping=True
        )
    
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

In [43]:
# Test sentences
test_sentences = [
    "Hello, how are you?",
    "What is your name?",
    "I love learning languages",
    "Good morning"
]

# Translate to each of your new languages
target_languages = ['pam', 'tao', 'pag', 'tl', 'en']

for sentence in test_sentences:
    print(f"\nOriginal: {sentence}")
    print("-" * 50)
    
    for lang in target_languages:
        translation = translate_to_language(sentence, lang)
        print(f"{lang}: {translation}")


Original: Hello, how are you?
--------------------------------------------------
pam: __tl__ Datapuwa't kung sinasabi ko sa inyo, ano ka na?
tao: __tl__ Datapuwa't sinasabi ko sa inyo, kayo'y kaniyang sinasabi?
pag: __tl__ Datapuwa't sinasabi ko sa inyo, kayo'y nakita?
tl: __en__ I tell you, you?"
en: __tl__ Sapagka't ano ang inyong mga bagay?

Original: What is your name?
--------------------------------------------------
pam: __tl__ At kung ikaw ang iyong pangalan?
tao: __tl__ At ang iyong pangalan?
pag: __tl__ At ang iyong pangalan?
tl: __en__ What you say?
en: __tl__ At ang iyong pangalan?

Original: I love learning languages
--------------------------------------------------
pam: __tl__ Datapuwa't nalalaman ko ang nag-aaroon ng mga tao.
tao: __tl__ Datapuwa't nalalaman ko ang nag-aaroon ng mga tao.
pag: __tl__ At nalalaman ko ang nag-aaroon sa kaniyang sarili'y nag-aaroon sa kanila,
tl: __en__ I tell you, I tell you,
en: __tl__ Ako'y nag-aaroon sa kaniyang sarili.

Original: Good

In [51]:
# Check if special tokens are properly recognized
print("Special tokens in tokenizer:")
print(tokenizer.all_special_tokens)
print("\nVocabulary size:", len(tokenizer))

# Check token IDs for your languages
for lang in ['pam', 'tao', 'pag', 'tl', 'en']:
    token = f"__{lang}__"
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"Token '{token}' -> ID: {token_id}")

# Test tokenization
test_text = "__pam__ Hello"
encoded = tokenizer(test_text, return_tensors="pt")
print(f"\nEncoded '{test_text}': {encoded['input_ids']}")
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Decoded back: {decoded}")

Special tokens in tokenizer:
['</s>', '<unk>', '<pad>', '__pam__', '__tao__', '__pag__']

Vocabulary size: 57376
Token '__pam__' -> ID: 57373
Token '__tao__' -> ID: 57374
Token '__pag__' -> ID: 57375
Token '__tl__' -> ID: 1
Token '__en__' -> ID: 1

Encoded '__pam__ Hello': tensor([[57373,    61, 50455,     0]])
Decoded back: __pam__ Hello</s>


In [52]:
test_text = "What is your name?"

print("Custom format (probably broken):")
for lang in ['pam', 'tao', 'pag']:
    source_text = f"__{lang}__ {test_text}"
    inputs = tokenizer(source_text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=128)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{lang}: {result}")

print("\nMarianMT format (might work):")
for lang in ['pam', 'tao', 'pag']:
    source_text = f">>{lang}<< {test_text}"
    inputs = tokenizer(source_text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=128)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{lang}: {result}")

Custom format (probably broken):
pam: __tl__ At kung ikaw ang iyong pangalan?
tao: __tl__ At ang iyong pangalan?
pag: __tl__ At ang iyong pangalan?

MarianMT format (might work):
pam: __tl__ At ano ang ikaw mo?
tao: __tl__ At ano ang ikaw mo?
pag: __tl__ At ano ang ikaw mo?
